# CTA historical service and ridership comparison

This notebook imports the eight dated CTA GTFS archives listed in the project manifest, aligns daily scheduled bus trips and trip-hours with CTA boardings by exact route and service date, then compares demand-only and service-aware regressors on the same matched rows.

Do not use the current google_transit.zip from CTA's schedule page for this historical comparison; it does not cover the ridership dates. Download each exact Mobility Database archive ID from the intern guide and place it in Google Drive as CTA_GTFS/<archive_id>/original.zip. The importer checks each archive SHA-256 against the guide.

The comparison is retrospective and rolling one-step. It does not claim the full test period could have been forecast from one earlier date.

In [ ]:
from pathlib import Path
import subprocess
from google.colab import drive, files
drive.mount('/content/drive')
REPO = Path('/content/Powered-Public-Transit-Route-Optimization')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'users/bidhubhusan',
                    'https://github.com/springboardmentor441p-coderr/Powered-Public-Transit-Route-Optimization.git', str(REPO)], check=True)
%cd /content/Powered-Public-Transit-Route-Optimization
%pip -q install -r requirements.txt

## 1. Locate ridership and archived schedules

The ridership CSV downloads from the official portal if it is not already in the Colab session. Put each schedule ZIP in the Drive folder shown below. Keep all eight snapshots separate.

In [ ]:
import pandas as pd
from urllib.request import urlretrieve
from src.gtfs_service import build_service_table
from src.compare_service_models import compare

RIDERSHIP_PATH = Path('/content/cta_bus_ridership.csv')
if not RIDERSHIP_PATH.exists():
    urlretrieve('https://data.cityofchicago.org/api/views/jyb9-n7fm/rows.csv?accessType=DOWNLOAD', RIDERSHIP_PATH)
GTFS_DRIVE_DIR = Path('/content/drive/MyDrive/CTA_GTFS')
manifest = pd.read_csv('data/raw/gtfs_archive_manifest.csv', dtype='string').fillna('')
manifest['zip_path'] = manifest['archive_id'].map(lambda archive_id: str(GTFS_DRIVE_DIR / archive_id / 'original.zip'))
MANIFEST_PATH = Path('/content/gtfs_archive_manifest_colab.csv')
manifest.to_csv(MANIFEST_PATH, index=False)
missing = [row.zip_path for row in manifest.itertuples() if not Path(row.zip_path).is_file()]
if missing:
    raise FileNotFoundError('Place all eight original.zip archives in Drive first. Missing:\n' + '\n'.join(missing))
print('Ridership file:', RIDERSHIP_PATH)
print('GTFS archive count:', len(manifest))

## 2. Import dated service and audit the exact join

The importer applies calendar_dates additions/removals, chooses the latest archive retrieved on or before each date for the whole network, and leaves unmatched rows visible. Stop-time tables are streamed, so this step can take several minutes.

In [ ]:
SERVICE_PATH = Path('data/processed/cta_route_date_demand_service.csv')
IMPORT_REPORT = Path('outputs/reports/gtfs_service_import.json')
import_report = build_service_table(RIDERSHIP_PATH, MANIFEST_PATH, SERVICE_PATH, IMPORT_REPORT)
print(pd.Series({k: v for k, v in import_report.items() if not isinstance(v, (dict, list))}))
print('Archive and date coverage details:')
display(pd.DataFrame(import_report['archive_versions']))
print('Unmatched ridership rows:', import_report['ridership_without_schedule_rows'])
print('Unmatched schedule rows:', import_report['schedule_without_ridership_rows'])

## 3. Compare demand-only and service-aware models

Both models use the same exact matched route/date rows and chronological partitions. Service-hour values remain missing when trip durations are incomplete; the model pipeline imputes them using training data and adds missingness indicators.

In [ ]:
COMPARISON_REPORT = Path('outputs/reports/service_model_comparison.json')
comparison = compare(RIDERSHIP_PATH, SERVICE_PATH, COMPARISON_REPORT, Path('models'))
print('Matched test coverage:', comparison['matched_test_coverage_pct'], '%')
print('Seasonal baselines on matched test rows:')
display(pd.DataFrame(comparison['seasonal_baselines_on_matched_test_rows']).T)
print('Model comparison:')
display(pd.DataFrame({name: result['test_metrics'] for name, result in comparison['models'].items()}).T)
print('Coverage, schedule versions, and detailed route/day-type errors are in the JSON report.')

## 4. Download the results

Save these reports with the project. The two fitted artifacts were trained on the same matched partitions and differ only in whether scheduled trips/service-hours are included.

In [ ]:
files.download(str(IMPORT_REPORT))
files.download(str(COMPARISON_REPORT))
files.download('models/cta_demand_only.joblib')
files.download('models/cta_service_aware.joblib')